# Ninai SDK — Streaming Ingestion

Demonstrates all streaming components:
- `EventIngestor` — fire-and-forget memory writes from any event dict
- `StreamPipeline` + custom adapter — plug any source into Ninai memory
- `NinaiEventStream` — consume live events from Ninai via WebSocket
- `NinaiSSEStream` — consume live events via SSE (no extra deps)
- `NinaiKafkaIngestor` — consume Kafka topics into Ninai memory

All cells run offline with mocks — no live server or broker needed.

In [1]:
import sys, os
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)

# nest_asyncio lets asyncio.run() work inside Jupyter's already-running event loop
import nest_asyncio
nest_asyncio.apply()

print('SDK path:', SDK_PATH)

SDK path: D:\Sansten\Projects\Ninai2\repos\ninai\sdk\python


## 1. EventIngestor — direct usage

`EventIngestor` maps a raw event dict to a `client.memories.create()` call.
No extra dependencies — uses the core `NinaiClient`.

> **Note:** `EventIngestor.ingest()` calls `memories.create` via `run_in_executor`
> (the SDK HTTP client is synchronous). The mock below uses a plain `MagicMock`.

In [2]:
import asyncio
from types import SimpleNamespace
from unittest.mock import MagicMock

from ninai.streaming.ingestor import EventIngestor

# memories.create is called synchronously inside run_in_executor — use plain MagicMock
mock_memory = SimpleNamespace(id='mem-001', content='Disk at 95%', tags=['prod', 'alert'])
mock_client = MagicMock()
mock_client.memories.create.return_value = mock_memory

ingestor = EventIngestor(mock_client, source_type='webhook', tags=['prod'])

event = {
    'title': 'Disk Alert',
    'content': 'Disk at 95% on prod-db-01',
    'severity': 'high',
    'source_id': 'alert-42',
}

result = asyncio.run(ingestor.ingest(event))
print('Ingested memory id:', result.id)
print('Tags:', result.tags)

# Verify create was called with the right keyword args
call_kwargs = mock_client.memories.create.call_args.kwargs
print('\ncreate() called with:')
for k, v in call_kwargs.items():
    print(f'  {k}: {v}')

Ingested memory id: mem-001
Tags: ['prod', 'alert']

create() called with:
  content: Disk at 95% on prod-db-01
  scope: organization
  memory_type: long_term
  classification: confidential
  source_type: webhook
  tags: ['prod']
  title: Disk Alert
  source_id: alert-42
  metadata: {'severity': 'high'}


## 2. Batch ingestion

In [3]:
batch_events = [
    {'title': 'CPU spike', 'content': 'CPU hit 98%', 'severity': 'critical', 'source_id': 'cpu-1'},
    {'title': 'Memory leak', 'content': 'RSS growing unbounded', 'severity': 'high', 'source_id': 'mem-2'},
    {'title': 'Deploy started', 'content': 'v2.4.1 deploy in progress', 'severity': 'low', 'source_id': 'deploy-3'},
]

mock_client2 = MagicMock()
mock_client2.memories.create.side_effect = [SimpleNamespace(id=f'mem-{i:03d}') for i in range(10)]

ingestor2 = EventIngestor(mock_client2, source_type='webhook', tags=['prod'])
results = asyncio.run(ingestor2.ingest_batch(batch_events))
print(f'Ingested {len(results)} events:')
for r in results:
    if r:
        print(f'  → {r.id}')

Ingested 3 events:
  → mem-000
  → mem-001
  → mem-002


## 3. Severity → classification mapping

`EventIngestor` automatically sets the memory `classification` based on event severity.

In [4]:
from ninai.streaming.ingestor import _SEVERITY_TO_CLASSIFICATION

print('Severity → classification mapping:')
for severity, classification in _SEVERITY_TO_CLASSIFICATION.items():
    print(f'  {severity:10s} → {classification}')

Severity → classification mapping:
  critical   → restricted
  high       → confidential
  medium     → internal
  low        → internal


## 4. StreamPipeline — custom adapter

`StreamPipeline` connects any `BaseStreamAdapter` to `EventIngestor`.
Build a minimal in-memory adapter that replays a fixed list of events.

In [5]:
from ninai.streaming.base import BaseStreamAdapter, StreamPipeline

class InMemoryAdapter:
    """Minimal adapter replaying a list of event dicts — no I/O needed."""

    def __init__(self, events):
        self._events = events

    async def __aenter__(self):
        print('InMemoryAdapter: connected')
        return self

    async def __aexit__(self, *_):
        print('InMemoryAdapter: disconnected')

    async def events(self):
        for ev in self._events:
            yield ev

print('BaseStreamAdapter protocol satisfied:', isinstance(InMemoryAdapter([]), BaseStreamAdapter))

pipeline_events = [
    {'content': 'Order #1001 placed', 'title': 'New order', 'severity': 'low'},
    {'content': 'Payment failed for #1002', 'title': 'Payment error', 'severity': 'high'},
    {'content': 'Refund processed for #999', 'title': 'Refund', 'severity': 'medium'},
]

mock_client3 = MagicMock()
mock_client3.memories.create.side_effect = [SimpleNamespace(id=f'mem-p{i:02d}') for i in range(10)]

ingestor3 = EventIngestor(mock_client3, source_type='custom', tags=['orders'])
adapter = InMemoryAdapter(pipeline_events)
pipeline = StreamPipeline(adapter=adapter, ingestor=ingestor3, concurrency=4)

asyncio.run(pipeline.run())
print('Pipeline stats:', pipeline.stats)

BaseStreamAdapter protocol satisfied: True
InMemoryAdapter: connected
InMemoryAdapter: disconnected
Pipeline stats: {'processed': 3, 'errors': 0}


## 5. NinaiSSEStream — Server-Sent Events consumer

Uses `httpx` streaming (always available — no extra install needed).
Consumes `/sse/events` and yields parsed event dicts.

In [6]:
import httpx
from contextlib import asynccontextmanager
from unittest.mock import patch
from ninai.streaming.sse import NinaiSSEStream

mock_sse_client = MagicMock()
mock_sse_client.base_url = 'https://api.ninai.ai/api/v1'
mock_sse_client._get_headers = lambda: {'Authorization': 'Bearer nai_test'}

# Three SSE events + one keepalive comment + one 'ping' (should be skipped)
sse_body = (
    'data: {"event_type": "memory.created", "id": "mem-101"}\n\n'
    ': keepalive\n\n'
    'data: {"event_type": "goal.updated", "id": "goal-55"}\n\n'
    'data: ping\n\n'
    'data: {"event_type": "memory.created", "id": "mem-102"}\n\n'
)

async def _run_sse():
    fake_resp = MagicMock()
    fake_resp.raise_for_status = MagicMock()

    async def _lines():
        for line in sse_body.split('\n'):
            yield line

    fake_resp.aiter_lines = _lines

    @asynccontextmanager
    async def _fake_stream(*a, **kw):
        yield fake_resp

    collected = []
    with patch.object(httpx.AsyncClient, 'stream', _fake_stream):
        stream = NinaiSSEStream(mock_sse_client)
        async for ev in stream:
            collected.append(ev)
    return collected

sse_events = asyncio.run(_run_sse())
print(f'Received {len(sse_events)} events (ping + keepalive skipped):')
for ev in sse_events:
    print(f'  {ev}')

Received 3 events (ping + keepalive skipped):
  {'event_type': 'memory.created', 'id': 'mem-101'}
  {'event_type': 'goal.updated', 'id': 'goal-55'}
  {'event_type': 'memory.created', 'id': 'mem-102'}


## 6. NinaiEventStream — WebSocket consumer

Requires `pip install "ninai[streaming]"` (websockets package).  
Connects to `/ws/stream`, reconnects on disconnect.

In [7]:
try:
    import websockets  # noqa: F401
    HAS_WEBSOCKETS = True
except ImportError:
    HAS_WEBSOCKETS = False

if not HAS_WEBSOCKETS:
    print('websockets not installed — skipping live demo.')
    print('Install with: pip install "ninai[streaming]"')
else:
    import json
    from unittest.mock import patch
    from ninai.streaming.websocket import NinaiEventStream

    mock_ws_client = MagicMock()
    mock_ws_client.base_url = 'https://api.ninai.ai/api/v1'
    mock_ws_client._get_headers = lambda: {'Authorization': 'Bearer nai_test'}

    ws_messages = [
        json.dumps({'event_type': 'memory.created', 'data': {'id': 'mem-200'}}),
        json.dumps({'event_type': 'goal.completed', 'data': {'id': 'goal-10'}}),
    ]

    class FakeWS:
        def __init__(self, messages):
            self._msgs = list(messages)
        async def __aenter__(self):
            return self
        async def __aexit__(self, *_):
            pass
        def __aiter__(self):
            return self._iter()
        async def _iter(self):
            for m in self._msgs:
                yield m
        async def send(self, msg):
            pass

    async def _run_ws():
        results = []
        stream = NinaiEventStream(mock_ws_client, max_reconnects=0)
        with patch('websockets.connect', return_value=FakeWS(ws_messages)):
            async for ev in stream:
                results.append(ev)
        return results

    ws_evs = asyncio.run(_run_ws())
    print(f'Received {len(ws_evs)} WebSocket events:')
    for ev in ws_evs:
        print(f'  {ev}')

Received 2 WebSocket events:
  {'event_type': 'memory.created', 'data': {'id': 'mem-200'}}
  {'event_type': 'goal.completed', 'data': {'id': 'goal-10'}}


## 7. NinaiKafkaIngestor — Kafka consumer

Requires `pip install "ninai[kafka]"` (aiokafka package).  
Consumes one or more Kafka topics and writes events into Ninai memory.

In [8]:
try:
    import aiokafka  # noqa: F401
    HAS_KAFKA = True
except ImportError:
    HAS_KAFKA = False

if not HAS_KAFKA:
    print('aiokafka not installed — skipping live demo.')
    print('Install with: pip install "ninai[kafka]"')
else:
    from unittest.mock import patch
    from ninai.streaming.kafka import NinaiKafkaIngestor

    kafka_client = MagicMock()
    kafka_client.memories.create.side_effect = [
        SimpleNamespace(id=f'kmem-{i:03d}') for i in range(10)
    ]

    ingestor_k = EventIngestor(kafka_client, source_type='kafka', tags=['prod', 'alerts'])

    # Custom transform: reshape Kafka message → Ninai event dict
    def incident_transform(raw: dict) -> dict:
        return {
            'title': raw.get('summary', 'Untitled'),
            'content': raw.get('description', ''),
            'severity': raw.get('priority', 'medium'),
            'source_id': raw.get('incident_id'),
            'tags': ['kafka', raw.get('service', 'unknown')],
        }

    fake_kafka_msgs = [
        SimpleNamespace(
            value={'summary': 'DB latency spike', 'description': 'p99 > 2s',
                   'priority': 'high', 'incident_id': 'INC-001', 'service': 'postgres'},
            offset=0,
        ),
        SimpleNamespace(
            value={'summary': 'API error rate', 'description': '5xx at 12%',
                   'priority': 'critical', 'incident_id': 'INC-002', 'service': 'api-gw'},
            offset=1,
        ),
    ]

    class FakeKafkaConsumer:
        def __init__(self, *args, **kwargs): pass
        async def start(self): print('Kafka consumer started')
        async def stop(self): print('Kafka consumer stopped')
        def __aiter__(self): return self._msgs()
        async def _msgs(self):
            for m in fake_kafka_msgs:
                yield m

    async def _run_kafka():
        with patch('ninai.streaming.kafka.AIOKafkaConsumer', FakeKafkaConsumer):
            async with NinaiKafkaIngestor(
                ingestor=ingestor_k,
                bootstrap_servers='kafka:9092',
                topic='incidents',
                group_id='ninai-demo',
                transform=incident_transform,
            ) as consumer:
                await consumer.run()
                return consumer.stats

    k_stats = asyncio.run(_run_kafka())
    print(f'Kafka ingestor stats: {k_stats}')

aiokafka not installed — skipping live demo.
Install with: pip install "ninai[kafka]"


## 8. Streaming extras summary

| Component | Import | Extra needed |
|-----------|--------|-------------|
| `EventIngestor` | `from ninai.streaming import EventIngestor` | none |
| `BaseStreamAdapter` | `from ninai.streaming import BaseStreamAdapter` | none |
| `StreamPipeline` | `from ninai.streaming import StreamPipeline` | none |
| `NinaiSSEStream` | `from ninai.streaming import NinaiSSEStream` | none |
| `NinaiEventStream` | `from ninai.streaming import NinaiEventStream` | `ninai[streaming]` |
| `NinaiKafkaIngestor` | `from ninai.streaming import NinaiKafkaIngestor` | `ninai[kafka]` |

Custom sources (Kinesis, Redis Streams, Pulsar, etc.) only need to implement the three-method `BaseStreamAdapter` protocol — no base class required.